In [1]:
from PIL import Image, ImageDraw

# Imagen RGB de 120x120 con color de fondo (crema)
img = Image.new("RGB", (120, 120), (240, 230, 200))
draw = ImageDraw.Draw(img)

draw.rectangle([15, 20, 55, 60], fill=(220, 30, 30))   # figura 1: rectángulo rojo
draw.ellipse([65, 70, 105, 110], fill=(30, 90, 200))   # figura 2: círculo azul

print(img.size, img.mode)
img.save("mi_imagen.png")

(120, 120) RGB


In [2]:
img = Image.open("mi_imagen.png")
print("Formato:", img.format)

# Coordenadas (x, y) que usarás también en la Tarea 2
p_fondo = (5, 5)
p_fig1  = (35, 40)   # dentro del rectángulo rojo
p_fig2  = (85, 90)   # dentro del círculo azul

print("Fondo   ", p_fondo, "->", img.getpixel(p_fondo))
print("Figura 1", p_fig1,  "->", img.getpixel(p_fig1))
print("Figura 2", p_fig2,  "->", img.getpixel(p_fig2))

Formato: PNG
Fondo    (5, 5) -> (240, 230, 200)
Figura 1 (35, 40) -> (220, 30, 30)
Figura 2 (85, 90) -> (30, 90, 200)


In [3]:
gris   = img.convert("L")
rotada = img.rotate(90, expand=True)
mitad  = img.resize((img.width // 2, img.height // 2))
volt   = img.transpose(Image.Transpose.FLIP_TOP_BOTTOM)

print("convert('L')  :", gris.size,   gris.mode)
print("rotate(90)    :", rotada.size)
print("resize (mitad):", mitad.size)
print("flip vertical :", volt.size)

convert('L')  : (120, 120) L
rotate(90)    : (120, 120)
resize (mitad): (60, 60)
flip vertical : (120, 120)


In [4]:
from PIL import ImageEnhance, ImageFilter

brillo_bajo = ImageEnhance.Brightness(img).enhance(0.5)   # factor menor a 1.0
brillo_alto = ImageEnhance.Brightness(img).enhance(2.5)   # factor mayor a 2.0

antes   = img.getpixel(p_fig1)
despues = brillo_alto.getpixel(p_fig1)
print("Píxel figura 1 ANTES  :", antes)
print("Píxel figura 1 DESPUÉS:", despues)
print("Canales recortados (0 o 255):", [c in (0, 255) for c in despues])

Píxel figura 1 ANTES  : (220, 30, 30)
Píxel figura 1 DESPUÉS: (255, 75, 75)
Canales recortados (0 o 255): [True, False, False]


In [5]:
blur   = img.filter(ImageFilter.GaussianBlur(radius=2))
bordes = img.filter(ImageFilter.FIND_EDGES)

blur.save("imagen_blur.png")
bordes.save("imagen_bordes.png")
print("Guardados: imagen_blur.png, imagen_bordes.png")

Guardados: imagen_blur.png, imagen_bordes.png


In [6]:
import numpy as np
from skimage import color, filters

arr = np.array(img)
print("Array:", arr.shape, arr.dtype)

gris_sk = color.rgb2gray(arr)          # float entre 0 y 1
x, y = p_fondo
valor_sk  = gris_sk[y, x] * 255        # en arrays el índice es [fila, columna] = [y, x]
valor_pil = gris.getpixel(p_fondo)     # convert("L") de Pillow

print("Pillow  convert('L')    :", valor_pil)
print("skimage rgb2gray * 255  :", round(valor_sk, 2))
print("Diferencia              :", round(abs(valor_sk - valor_pil), 2))

Array: (120, 120, 3) uint8
Pillow  convert('L')    : 230
skimage rgb2gray * 255  : 229.96
Diferencia              : 0.04


In [7]:
umbral  = filters.threshold_otsu(gris_sk)
binaria = gris_sk > umbral

print("Umbral de Otsu         :", round(umbral, 4), "(≈", round(umbral * 255, 1), "en escala 0-255)")
print("Proporción de True     :", round(binaria.mean(), 4))

Umbral de Otsu         : 0.3334 (≈ 85.0 en escala 0-255)
Proporción de True     : 0.8833
